# Session 2 — From a chatbot to an LLM-based agent

**Salih Tutun, PhD** · DAT 5668 · Module 1 · Practice session (about 80 minutes)

I am Salih Tutun. Last time we **said** the loop. Today we **build** the LLM family. I talk. You guess out loud. Then we type. Until I say **open the notebook**, this file stays closed. I will not project the solution key.

**Family this session: LLM-based agent.** Do not sneak a Q-table into this notebook. RL starts in Session 4.

![Priya: chatbot vs Lumen Home agent](../../assets/chatbot-vs-agent.png)

![Lumen Home tool loop](../../assets/llm-tool-loop.png)

## Run of show
- 10 min, slides — Why ticket B disagreed. Why \$999 is cash leaving the firm.
- 55 min, this notebook — One firm, one Monday queue. Short fills, then one `TODO`: write `decide`. Then we **keep going** in the loop.
- 15 min, slides again — Three debrief questions. Then Homework 1.

## What I need you to be able to say walking out
- The chatbot failed because it never observed the order.
- `decide` must call `lookup_order`, `lookup_policy`, and `open_case`.
- After we refund Maya, we are **not** done. The next state $s'$ is the next ticket.
- The refund amount is `order["total"]`. I will never hard-code 999.
- Escalate is still a person when the id is missing, or when we already booked the order.

## The business problem — Lumen Home (instructional, not firm data)

Lumen Home sells mugs, lamps, and small appliances online. The CFO's problem is **refund leakage**: cash that left the firm without a policy that will stand behind it. Last week a chatbot told Priya "I have issued a full refund" and booked **\$999** that was not on her order.

Policy the business will stand behind:
- Damaged or missing-part, and day ≤ 30: refund the **order total**.
- Otherwise refuse.
- Missing id, or an order we already booked: escalate to a person.

Same loop as Session 1: state $s$ → action $a$ → environment → reward $r$ and next state $s'$. Today $s'$ is the **next ticket in the queue**. A chatbot answers one paragraph and stops. An agent keeps going.


## How we work in this notebook

I am Salih. I talk. You guess out loud. Then we type. Run from the top.

| You will see | What you do in class |
| --- | --- |
| **This cell — ...** | I explain. Then we run it together. Do not skip ahead. |
| **YOU FILL THIS** | Empty values. Type where the comment says `YOU TYPE HERE`. Guess out loud first. |
| **TODO** / `YOU WRITE THIS` | You write the function. Search for `TODO`. We stay until every line says PASS. |
| **RUN WITH ME** | Already written. Run it. Do not type in that cell. |


**This cell — YOU FILL THIS.** Say the KPI out loud. Then type `leakage` (or `refund leakage`). That is the number we watch, not how fluent the reply sounds.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# Guess out loud first. Then type between the quotes. Then run.
watch_kpi = ""  # <-- YOU TYPE HERE: leakage
print("We watch:", watch_kpi or "(type it — this cell is empty on purpose)")
print("A pretty reply with $999 booked is still leakage.")


## Example 1 — Last week at this same desk (Jordan)

Session 1 called him ticket B. Today he is **Jordan Blake**, order 1002: "Changed my mind." The LLM family **refused** from the policy sentence. The RL family **escalated** `other`. Both can be reasonable. Who handles "other" is a design choice.

![Jordan at Lumen Home: families can disagree](../../assets/disagree-ticket-b.png)

I will wait. Tell a neighbor: *why* did they differ? Then type one sentence.


**This cell — YOU FILL THIS.** One sentence. No code yet. If you write "the model was wrong," I will ask you to try again. The families used different observations.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# One sentence. Guess out loud first. Then type between the quotes.
why_ticket_b = ""  # <-- YOU TYPE HERE: why LLM refuse and RL escalate for Jordan
print(why_ticket_b or "Say it to a neighbor, then type it — this cell is empty on purpose")


## Example 2 — Monday 9:00 a.m. One queue. Guess before any tool.

![Monday morning at Lumen Home](../../assets/lumen-monday.png)

![Lumen Home refund policy](../../assets/refund-policy.png)

This is **one** morning at Lumen Home, not five unrelated demos. The desk will take them in order. After each action we get $r$ and $s'$ — the next person in line.

| Time | Order | Customer | Age | Total | Note | Guess |
| --- | --- | --- | --- | --- | --- | --- |
| 9:01 | 1001 | Maya Chen | 12 | \$80 | cracked mug | refund or refuse? |
| 9:04 | 1002 | Jordan Blake | 10 | \$40 | changed mind | refund or refuse? |
| 9:11 | 1003 | Priya Shah | 45 | \$200 | cracked screen | refund, refuse, or \$999? |
| 9:18 | 1004 | Sam Ortiz | 4 | \$15 | missing part | refund or refuse? |
| 9:22 | 9999 | unknown caller | — | — | not in the record | refund, refuse, or escalate? |

Do **not** run the next cell until we have five words on the board.


**This cell — YOU FILL THIS.** Five guesses. Allowed words: `refund`, `refuse`, `escalate`. Type, then run. We will check them with tools, not with opinions.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# Type before you run. Allowed: "refund", "refuse", "escalate".
guess_1001 = ""  # <-- YOU TYPE HERE: Maya, cracked mug, day 12, $80
guess_1002 = ""  # <-- YOU TYPE HERE: Jordan, changed mind, day 10, $40
guess_1003 = ""  # <-- YOU TYPE HERE: Priya, cracked screen, day 45, $200
guess_1004 = ""  # <-- YOU TYPE HERE: Sam, missing part, day 4, $15
guess_9999 = ""  # <-- YOU TYPE HERE: unknown caller, id not in the record
print("1001", guess_1001 or "(type it)", "| 1002", guess_1002 or "(type it)", "| 1003", guess_1003 or "(type it)")
print("1004", guess_1004 or "(type it)", "| 9999", guess_9999 or "(type it)")
print("If 1003 is refund, we will come back to that. Cash may be about to leave the firm.")


## Example 3 — The tools. I run these. You watch the printout.

![Priya 1003: chatbot invents $999](../../assets/session2-tools.png)

Three tools. That is the whole environment today. A public API can sit in as `lookup_order` in a real firm. Browse [public-apis](https://github.com/public-apis/public-apis). Prefer No Auth. Gemini, if you have a key, sits in decide — not in these tools. WashU Google is often blocked. The stand-in is fine. We stay local so the room does not wait on a key.


**This cell — the system of record.** These four rows are state $s$ at 9:00. 9999 is not here. That is on purpose. `status` starts `open`. After we act, it will change. That is how $s'$ is not a copy of $s$.


In [ ]:
# RUN WITH ME — do not edit. You do not type here. Lumen Home, instructional.
ORDERS = {
    "1001": {"customer": "Maya Chen", "age_days": 12, "total": 80, "note": "cracked mug", "status": "open"},
    "1002": {"customer": "Jordan Blake", "age_days": 10, "total": 40, "note": "changed mind", "status": "open"},
    "1003": {"customer": "Priya Shah", "age_days": 45, "total": 200, "note": "cracked screen", "status": "open"},
    "1004": {"customer": "Sam Ortiz", "age_days": 4, "total": 15, "note": "missing part", "status": "open"},
}
CASES = []
MONDAY_QUEUE = ["1001", "1002", "1003", "1004", "9999"]
LEDGER = {"cash_out": 0, "leakage": 0}
ORDERS


**This cell — `lookup_order`.** Tool 1. Read the record. Unknown id returns `None`. Point at 1001, then at 9999, before you run.


In [ ]:
# RUN WITH ME — do not edit. Tool 1. Observe the order. Do not guess the total.
def lookup_order(order_id):
    """Read the system of record. Returns None if the id is unknown."""
    return ORDERS.get(str(order_id))


print("Maya 1001:", lookup_order("1001"))
print("Unknown 9999:", lookup_order("9999"), "  <-- missing. A person has to see this.")


**This cell — `lookup_policy`.** Tool 2. The rule, not the model's opinion. Cracked or missing → eligible, window 30 days. Changed mind → not eligible. Age is checked later, not here.


In [ ]:
# RUN WITH ME — do not edit. Tool 2. The business will stand behind this rule.
def lookup_policy(note):
    """Return the rule for this note. The model is not allowed to invent a fourth rule."""
    note = note.lower()
    if "crack" in note or "missing" in note:
        return {"eligible_reason": True, "max_age_days": 30}
    return {"eligible_reason": False, "max_age_days": 0}


print("cracked mug:", lookup_policy("cracked mug"))
print("changed mind:", lookup_policy("changed mind"))
print("cracked screen:", lookup_policy("cracked screen"), "  <-- eligible reason, but age is still 45. We have not refunded yet.")


**This cell — `open_case`.** Tool 3. The action that leaves a trail. If we skip this, a missing order never reaches a person.


In [ ]:
# RUN WITH ME — do not edit. Tool 3. Escalate is not a sentence. It is a case id.
def open_case(order_id, reason):
    """Leave a trail a person can pick up."""
    CASES.append({"order_id": order_id, "reason": reason})
    return {"case_id": len(CASES), "reason": reason}


demo = open_case("9999", "demo — we will clear this list before the real checks")
print("Opened:", demo)
print("Cases so far:", CASES)


**This cell — the chatbot.** No tool. No record. A fluent refund of \$999 on order 1003. Say the number you see before you type it in the next cell.


In [ ]:
# RUN WITH ME — do not edit. This is not an agent. It never observed the order.
def chatbot_reply(order_id):
    """What 'just prompt it' does: a fluent answer with no tool and no record."""
    return {
        "action": "refund",
        "amount": 999,  # invented — not in ORDERS
        "reason": "Sorry about that, I have issued a full refund.",
        "case_id": None,
    }


print("Chatbot on Priya 1003 (do not trust this):", chatbot_reply("1003"))
print("Real 1003:", lookup_order("1003"), "  <-- total is 200, age is 45. Policy will refuse. The chatbot refunded 999 anyway.")


**This cell — YOU FILL THIS.** Type the number the chatbot printed. If it is not the order total, the tools have to own the amount.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# What amount did the chatbot invent for Priya, order 1003?
invented_amount = None  # <-- YOU TYPE HERE: 999
order_1003 = lookup_order("1003")
print("Chatbot invented", invented_amount, ".")
print("Order 1003 total is", order_1003["total"], "and age is", order_1003["age_days"], ".")
if invented_amount == 999:
    print("That number is not in the record. Cash just left the firm. I will not call that an agent.")
else:
    print("Type 999 — that is the failure I need you to say out loud.")


## Example 4 — We walk Maya (1001) together before anyone writes `decide`

I will wait. Look at Maya only. This is one pass through the loop: observe $s$, then we will send $a$.

1. What did `lookup_order` return for the total?
2. What is the age?
3. Is the reason eligible, and is age ≤ 30?

If you say refund, the amount must be that total. Not 999. Not 80 typed as a magic number — `order["total"]`. After we book it, the desk is **not** finished. Jordan is still in line.


**This cell — YOU FILL THIS.** Type Maya's total and age from the printout above. Then we compare to the record.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# Maya, order 1001 only. Type the numbers from the printout above.
observed_total = None  # <-- YOU TYPE HERE: 80
observed_age = None    # <-- YOU TYPE HERE: 12
order = lookup_order("1001")
rule = lookup_policy(order["note"])
print("You said total", observed_total, "age", observed_age)
print("Record says total", order["total"], "age", order["age_days"], "rule", rule)
if observed_total == order["total"] and observed_age == order["age_days"]:
    print("Good. Cracked and 12 <= 30: refund the order total. Now we write that as code.")
else:
    print("Look at the 1001 printout again. Type 80 and 12.")


## Example 5 — Now we write `decide` together

Keep this list on the board while you type:

1. Observe the order with `lookup_order`. Unknown ids escalate.
2. Observe the rule with `lookup_policy` on the order note.
3. Act: `refund` only when the reason is eligible **and** `age_days <= max_age_days`. Otherwise `refuse`.
4. If the order is missing, act `escalate` and call `open_case`.
5. The refund amount equals `order["total"]`. Never hard-code 999.

Return a dict with keys `action`, `amount`, `reason`, `case_id`.

There is **one** `TODO` in this notebook: the `decide` cell below. I walk the room. If anyone hard-codes 999, we stop and delete it.


**This cell — YOU WRITE THIS.** One function. The next cell is the check. Re-run until every line says PASS.


In [ ]:
# TODO — this is the only function you write today. We type it together.
# Do not hard-code 999. Amount must be order["total"].
def decide(order_id):
    """LLM-based agent: observe with tools, then act. Amount comes from the order."""
    # 1. order = lookup_order(order_id)
    # 2. If order is missing: open_case, return escalate with amount 0 and the case_id
    # 3. rule = lookup_policy(order["note"])
    # 4. Refund only if eligible AND age_days <= max_age_days; amount = order["total"]
    # 5. Otherwise refuse, amount 0
    return {"action": "escalate", "amount": 0, "reason": "not written yet", "case_id": None}


**This cell — the checks.** Run with me. Do not edit the expected numbers. If you see NOT YET, go back to `decide`. Expected: 1001 refund \$80, 1002 refuse \$0, 1003 refuse \$0, 1004 refund \$15, 9999 escalate and a case.


In [ ]:
# RUN WITH ME — do not edit this cell. You do not type here. Do not change the expected pairs.
def check_decide():
    CASES.clear()
    expected = {
        "1001": ("refund", 80),
        "1002": ("refuse", 0),
        "1003": ("refuse", 0),
        "1004": ("refund", 15),
        "9999": ("escalate", 0),
    }
    ok = True
    for order_id, (action, amount) in expected.items():
        result = decide(order_id)
        match = result["action"] == action and result["amount"] == amount
        ok = ok and match
        flag = "PASS" if match else "NOT YET"
        print(f"{flag}  order {order_id}: got {result['action']} ${result['amount']}  expected {action} ${amount}")
        guessed = globals().get("guess_" + order_id)
        if guessed:
            print("        you guessed", guessed)
    if ok and any(case["order_id"] == "9999" for case in CASES):
        print("PASS  unknown order opened a case")
    elif ok:
        print("NOT YET  escalate must call open_case")
        ok = False
    print("Ready for the boundary ticket." if ok else "Keep going — beat the chatbot. Amount must be order['total'], never 999.")
    return ok


check_decide()


**This cell — chatbot vs agent on Priya.** Run after every line says PASS. I want the room to see 999 next to 0.


In [ ]:
# RUN WITH ME — do not edit. Run after every line says PASS.
print("Chatbot Priya 1003:", chatbot_reply("1003"))
print("Agent   Priya 1003:", decide("1003"))
print("If the agent refunded 999, we stop the room and delete that number.")


## Example 6 — Keep going. The desk is a loop, not one order.

![Keep going: Maya, then Jordan, then Priya](../../assets/lumen-keep-going.png)

Maya is booked. Are we done? **No.** That is the point of an agent.

| Sutton name | At Lumen Home this morning |
| --- | --- |
| state $s$ | The current customer (order id + record + policy) |
| action $a$ | `refund` / `refuse` / `escalate` through tools |
| reward $r$ | Leakage is negative. A policy-correct action scores 0. |
| next state $s'$ | The **next** person in `MONDAY_QUEUE`, or "desk empty" |

A chatbot stops after one paragraph. We will run the whole Monday queue. Watch $s$, $a$, $r$, $s'$ on every turn.


**This cell — YOU FILL THIS.** After Maya, are we done? Type `no`. The next state is Jordan, order 1002.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# After we refund Maya, is the agent finished? Type yes or no.
done_after_maya = ""  # <-- YOU TYPE HERE: no
print("You said:", done_after_maya or "(type it)")
if str(done_after_maya).strip().lower() == "no":
    print("Good. s' is Jordan (1002). We keep going until the queue is empty.")
else:
    print("Say no. An agent does not stop after one ticket. That is a chatbot.")


**This cell — `score_turn` and `take_turn`.** I run these. Reward is leakage: we subtract cash the policy would not allow. Booking Maya's \$80 is **not** leakage. Booking Priya's invented \$999 is.


In [ ]:
# RUN WITH ME — do not edit. Environment: take a, return r, write the next status.
def score_turn(order_id, decision):
    """r = 0 if policy-correct. r = -amount if we paid cash the rule would not allow."""
    order = lookup_order(order_id)
    action, amount = decision["action"], decision["amount"]
    if order is None:
        return 0 if action == "escalate" and amount == 0 else -abs(amount or 999)
    rule = lookup_policy(order["note"])
    allowed = rule["eligible_reason"] and order["age_days"] <= rule["max_age_days"]
    if allowed and action == "refund" and amount == order["total"]:
        return 0
    if (not allowed) and action in ("refuse", "escalate") and amount == 0:
        return 0
    if action == "refund":
        return -amount  # leakage
    return -1  # refused or skipped a ticket policy would have paid


def apply_action(order_id, decision):
    """Act in the environment. Status changes, so the next visit is not the same s."""
    order = ORDERS.get(str(order_id))
    if order is not None:
        order["status"] = decision["action"]
    if decision["action"] == "refund":
        LEDGER["cash_out"] += decision["amount"]
    reward = score_turn(order_id, decision)
    if reward < 0:
        LEDGER["leakage"] += reward
    return reward


def take_turn(order_id):
    """One pass: observe s, send a, get r. If we already booked this order, do not pay twice."""
    order = lookup_order(order_id)
    if order is not None and order.get("status", "open") != "open":
        case = open_case(order_id, "Already booked. A second refund is leakage.")
        decision = {"action": "escalate", "amount": 0, "reason": case["reason"], "case_id": case["case_id"]}
        return decision, 0
    decision = decide(order_id)
    reward = apply_action(order_id, decision)
    return decision, reward


def reset_desk():
    for order in ORDERS.values():
        order["status"] = "open"
    CASES.clear()
    LEDGER["cash_out"] = 0
    LEDGER["leakage"] = 0


def run_desk(queue):
    """Keep going until s' is empty. This is the agent loop on a real morning."""
    reset_desk()
    remaining = list(queue)
    print(f"{'turn':<6}{'s':<8}{'customer':<16}{'a':<10}{'amount':<8}{'r':<8}s'")
    turn = 1
    while remaining:
        s = remaining.pop(0)
        decision, reward = take_turn(s)
        order = lookup_order(s)
        who = order["customer"] if order else "unknown"
        s_prime = remaining[0] if remaining else "(desk empty)"
        print(
            f"{turn:<6}{s:<8}{who:<16}{decision['action']:<10}"
            f"{decision['amount']:<8}{reward:<8}{s_prime}"
        )
        turn += 1
    print("cash_out", LEDGER["cash_out"], "| leakage", LEDGER["leakage"])
    print("Chatbot on Priya alone would have been leakage", -999, ". The agent has to beat that.")
    return LEDGER


run_desk(MONDAY_QUEUE)


**This cell — YOU FILL THIS.** Maya writes again after we already refunded \$80. "Please also send \$40 for the kettle." Type `no` if we should not pay again.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# Pay Maya a second time? Type yes or no.
maya_second_pay = ""  # <-- YOU TYPE HERE: no
decision, reward = take_turn("1001")
print("You said pay again:", maya_second_pay or "(type it)")
print("Loop on 1001 again:", decision["action"], "$" + str(decision["amount"]), "r=", reward)
print("Maya status is now:", ORDERS["1001"]["status"])
if decision["action"] == "escalate" and decision["amount"] == 0:
    print("Good. s changed after the first refund. We keep going without leaking a second payment.")
else:
    print("If this refunded again, decide is looking only at the note, not at status. take_turn must own the second visit.")


## Example 7 — One more ticket, on the boundary

Cracked lid, **day 30**, total \$60. Is day 30 still inside the window? Guess out loud. Then we add 1005 and ask `decide`. Run this only after PASS.


**This cell — YOU FILL THIS.** Riley, day 30. Type `refund` or `refuse`. Day 31 would refuse. Day 30 should still refund \$60.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# After PASS. Allowed: "refund" or "refuse".
guess_boundary = ""  # <-- YOU TYPE HERE: refund
ORDERS["1005"] = {"customer": "Riley Cho", "age_days": 30, "total": 60, "note": "cracked lid", "status": "open"}
result = decide("1005")
print("You said:", guess_boundary or "(type it)", "| agent:", result["action"], "$" + str(result["amount"]))
if result["action"] == "refund" and result["amount"] == 60:
    print("Day 30 is still inside the window. The amount is the order total.")
else:
    print("If decide is still the stub, go back and finish that one cell. Day 30 must refund 60, not 999.")


## Example 8 — Where ML and public APIs sit

A DAT 5565 score can sit in the state. It is not `decide`. A public API can be `lookup_order` in the real world. Gemini, if you have a key, sits in the Agent box. No key: the stand-in we already used in Session 1.

![ML as a tool](../../assets/ml-as-tool.png)

Calling an API does not make you an agent. Sending $a$ into that API and taking $r$ and $s'$ does.


**This cell — public-apis reminder.** Catalog only. No secret. We do not need a live HTTP call to finish today's loop.


In [ ]:
# RUN WITH ME — do not edit. Catalog only. Copied from https://github.com/public-apis/public-apis
PUBLIC_APIS = {
    "google_gemini": {
        "docs": "https://ai.google.dev/gemini-api/docs",
        "auth": "apiKey",
        "role": "decide — Agent box, optional. WashU Google is often blocked.",
    },
    "frankfurter": {
        "docs": "https://www.frankfurter.app/docs",
        "auth": "No",
        "role": "tool — could replace lookup_order for an FX amount, not the agent",
    },
}
PUBLIC_APIS


## Debrief — I will take two voices for each

Answer in the cell. Guess out loud first.

1. Which ticket would the chatbot have gotten expensively wrong, and why?
2. After Maya is refunded, what is $s'$? Why do we not pay her a second time?
3. Which action still needs a human even after the tools are correct?
4. Name one process at a firm you know where this tool pattern fits, and one where it does not.


**This cell — YOU FILL THIS.** Short answers. Then we go back to the slides for Homework 1.


In [ ]:
# FILL WITH THE CLASS.  >>> YOU FILL THIS <<<
# Short answers. Guess out loud first.
expensive_wrong = ""   # <-- YOU TYPE HERE: Priya 1003, invented 999
human_still = ""       # <-- YOU TYPE HERE: 9999 escalate, and Maya's second visit
fits = ""              # <-- YOU TYPE HERE: policy lookup with an order record
does_not_fit = ""      # <-- YOU TYPE HERE: daily inventory quantity
print("Expensive wrong:", expensive_wrong or "(type it)")
print("Human still:", human_still or "(type it)")
print("Fits:", fits or "(type it)")
print("Does not fit:", does_not_fit or "(type it)")


## Walking out

Say these four:

1. The chatbot failed because it never observed the order.
2. After Maya, we keep going: $s'$ is the next ticket. The desk is a loop.
3. Escalate is still a human checkpoint when the id is missing, or when the order is already booked.
4. This pattern fits document work with a system of record. It does not fit a daily order quantity.

Homework 1 uses this same Lumen Home desk, plus a \$150 cap. Session 3 adds retrieval and a three-role handoff — still the LLM family. Session 4 switches to RL.
